<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Model Routing & Cost Control
</b></font> </br></p>

---


**Beitrag zum Leitprojekt:** M25 ist die Betriebsschicht zwischen Evaluation (M24) und Integration (M26). Der Meeting- & Research-Briefing-Agent soll nicht blind immer dasselbe Modell aufrufen, sondern Modellwahl, Fallback, Tokenverbrauch und Budgetstatus nachvollziehbar machen. Damit wird aus einer guten Demo ein kontrollierbarer Agentenbaustein für Integration und Deployment.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M25-Model-Routing-Cost-Control"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

from langchain.chat_models import init_chat_model
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

MODEL_CHAIN = [ROUTER, WORKER, JUDGE]

run_cfg = {
    "run_name": "M25_Model_Routing_Cost_Control",
    "tags": ["m25", "routing", "cost-control"],
    "metadata": {"notebook": "M25", "version": "1.0"},
}


# 1 | Übersicht

| Betriebsfrage | Pattern in diesem Modul | Anschluss im Kurs |
|---|---|---|
| Welches Modell bearbeitet diese Anfrage? | Model Chain mit Routing-Reihenfolge | M26 Integration Pipeline |
| Was passiert bei Ausfall oder Rate Limit? | Fehlerklassifikation und Circuit Breaker | M36 Production Deployment |
| Wie teuer war der Aufruf? | `usage_metadata` auswerten | M24 Evaluation, M37 API Deployment |
| Darf der Agent weitermachen? | Budget Gate | M23 Security, M24 Quality Gate |


In [ ]:
#@markdown <p><font size="4" color='green'>Routing- und Budget-Flow</font></p>

diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    A[Briefing-Auftrag] --> B[Model Router]
    B --> C{Modell verfügbar?}
    C -- ja --> D[LLM-Aufruf]
    C -- nein --> E[Nächstes Modell]
    E --> B
    D --> F[usage_metadata]
    F --> G{Budget ok?}
    G -- ja --> H[Antwort freigeben]
    G -- nein --> I[Review oder sparsamere Strategie]
"""
mermaid(diagram, width=900)

# 2 | Fehlerklassifikation und Circuit Breaker

Nicht jeder Fehler bedeutet dasselbe. Ein falscher API-Key ist ein eigener Konfigurationsfehler. Ein Rate Limit ist häufig transient. Ein nicht erreichbares Modell sollte vorübergehend aus der Rotation genommen werden.

Der Circuit Breaker merkt sich solche Ausfälle für eine kurze Zeit. Dadurch versucht der Agent nicht bei jeder Anfrage erneut ein Modell, das gerade offensichtlich nicht funktioniert.


Für Evaluation ist die Kategorie `OURS` besonders wichtig: Sie steht für Fehler, die durch Konfiguration, Berechtigungen oder falsche Eingaben im eigenen System entstehen. Solche Ereignisse sind keine Modellqualität, sondern Betriebs- oder Security-Signale und sollten in M24 als Regressionstor ausgewertet werden.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from time import monotonic
from typing import Any


class ModelUnavailableError(RuntimeError):
    """Raised in demos to simulate a provider/model outage."""


class ErrorCategory(str, Enum):
    DEAD = "dead"
    TRANSIENT = "transient"
    OURS = "ours"


@dataclass
class CircuitBreaker:
    cooldown_seconds: float = 30.0
    dead_until: dict[str, float] = field(default_factory=dict)

    def is_open(self, model: str) -> bool:
        return self.dead_until.get(model, 0.0) > monotonic()

    def mark_dead(self, model: str) -> None:
        self.dead_until[model] = monotonic() + self.cooldown_seconds

    def status(self) -> dict[str, float]:
        now = monotonic()
        return {model: round(until - now, 1) for model, until in self.dead_until.items() if until > now}


def get_status_code(error: Exception) -> int | None:
    response = getattr(error, "response", None)
    return getattr(response, "status_code", None)


def classify_error(error: Exception) -> ErrorCategory:
    status_code = get_status_code(error)

    if isinstance(error, ModelUnavailableError):
        return ErrorCategory.DEAD
    if status_code in {404, 410, 429, 500, 502, 503, 504}:
        return ErrorCategory.DEAD if status_code in {404, 410} else ErrorCategory.TRANSIENT
    if status_code in {400, 401, 403}:
        return ErrorCategory.OURS
    return ErrorCategory.TRANSIENT

# 3 | Router implementieren

Der Router arbeitet bewusst einfach:

1. Modelle werden in Reihenfolge versucht.
2. Offene Circuit-Breaker werden übersprungen.
3. Bei transienten Fehlern gibt es wenige Wiederholungen.
4. Bei toten Modellen wird auf das nächste Modell gewechselt.
5. Jede Entscheidung landet im Trace.


In [ ]:
MAX_ATTEMPTS = 2


def trace_event(trace: list[dict[str, Any]], model: str, event: str, detail: str | None = None) -> None:
    trace.append({"model": model, "event": event, "detail": detail})


def call_llm(
    model: str,
    prompt: str,
    forced_dead_models: set[str] | None = None,
) -> tuple[str, dict[str, Any]]:
    if forced_dead_models and model in forced_dead_models:
        raise ModelUnavailableError(f"Demo-Ausfall für {model}")

    llm = init_chat_model(model)
    response = llm.invoke(prompt, config=run_cfg)
    usage_metadata = getattr(response, "usage_metadata", None) or {}
    return response.content, usage_metadata


def route_llm(
    prompt: str,
    models: list[str],
    breaker: CircuitBreaker,
    forced_dead_models: set[str] | None = None,
) -> dict[str, Any]:
    trace: list[dict[str, Any]] = []

    for model in models:
        if breaker.is_open(model):
            trace_event(trace, model, "skip", "Circuit Breaker offen")
            continue

        for attempt in range(1, MAX_ATTEMPTS + 1):
            trace_event(trace, model, "try", f"Versuch {attempt}")

            try:
                content, usage_metadata = call_llm(model, prompt, forced_dead_models)
                trace_event(trace, model, "ok", "Antwort erhalten")
                return {
                    "content": content,
                    "served_by": model,
                    "usage_metadata": usage_metadata,
                    "trace": trace,
                    "breaker_status": breaker.status(),
                }

            except Exception as error:
                category = classify_error(error)
                trace_event(trace, model, category.value, str(error))

                if category == ErrorCategory.OURS:
                    raise
                if category == ErrorCategory.DEAD:
                    breaker.mark_dead(model)
                    break

    raise RuntimeError("Kein Modell konnte die Anfrage bearbeiten.")


def show_router_trace(result: dict[str, Any]) -> None:
    for step in result["trace"]:
        detail = f" - {step['detail']}" if step.get("detail") else ""
        print(f"{step['model']}: {step['event']}{detail}")

# 4 | Demo: Briefing-Anfrage mit kontrolliertem Fallback

Für die Demo wird das erste Modell künstlich als nicht verfügbar markiert. So sieht man den Fallback im Trace, ohne auf einen echten Provider-Ausfall warten zu müssen.


In [ ]:
briefing_prompt = """
Du bist ein Assistent für ein Meeting-Briefing.
Erstelle in maximal fünf Sätzen ein Kurzbriefing für ein Teammeeting zur Einführung eines RAG-Systems.
Nenne Ziel, Risiko und eine sinnvolle nächste Prüfung.
""".strip()

breaker = CircuitBreaker(cooldown_seconds=60)
forced_dead_models = {MODEL_CHAIN[0]}

router_result = route_llm(
    briefing_prompt,
    models=MODEL_CHAIN,
    breaker=breaker,
    forced_dead_models=forced_dead_models,
)

mprint(router_result["content"])
print("\nServed by:", router_result["served_by"])

In [ ]:
show_router_trace(router_result)
breaker.status()

In [ ]:
fallback_used = router_result["served_by"] != MODEL_CHAIN[0]
dead_detected = any(step["event"] == ErrorCategory.DEAD.value for step in router_result["trace"])

assert fallback_used, "Die Demo sollte auf ein anderes Modell ausweichen."
assert dead_detected, "Der simulierte Ausfall sollte im Trace sichtbar sein."

print("Fallback genutzt:", fallback_used)
print("Ausfall erkannt:", dead_detected)

# 5 | Token und Kosten aus `usage_metadata` berechnen

LangChain-Nachrichten enthalten häufig `usage_metadata`. Die genaue Struktur kann je nach Provider variieren. Für Kurszwecke nutzen wir die üblichen Felder `input_tokens`, `output_tokens`, `total_tokens` und optional `input_token_details.cache_read`.

Die folgenden Preise sind **Demo-Werte**. Sie dienen dazu, die Rechnung zu verstehen. Für echte Budgets müssen die Preise aus einer zentral gepflegten Konfiguration kommen.


In [ ]:
DEMO_PRICES_PER_1M_TOKENS = {
    "gpt-5.6-luna": {"input": 0.05, "cached_input": 0.005, "output": 0.40},
    "gpt-5.4-mini": {"input": 0.25, "cached_input": 0.025, "output": 2.00},
    "gpt-5.4": {"input": 1.25, "cached_input": 0.125, "output": 10.00},
}


def model_short_name(model_id: str) -> str:
    return model_id.split(":", maxsplit=1)[-1]


def calculate_call_cost(usage_metadata: dict[str, Any], model_id: str) -> dict[str, Any]:
    short_name = model_short_name(model_id)
    prices = DEMO_PRICES_PER_1M_TOKENS.get(short_name)

    input_tokens = int(usage_metadata.get("input_tokens", 0) or 0)
    output_tokens = int(usage_metadata.get("output_tokens", 0) or 0)
    total_tokens = int(usage_metadata.get("total_tokens", input_tokens + output_tokens) or 0)

    input_details = usage_metadata.get("input_token_details", {}) or {}
    cached_input_tokens = int(input_details.get("cache_read", 0) or 0)
    billable_input_tokens = max(input_tokens - cached_input_tokens, 0)

    if prices is None:
        return {
            "model": model_id,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "cached_input_tokens": cached_input_tokens,
            "cost_usd": None,
            "note": "Kein Demo-Preis für dieses Modell hinterlegt.",
        }

    cost_usd = (
        billable_input_tokens * prices["input"]
        + cached_input_tokens * prices["cached_input"]
        + output_tokens * prices["output"]
    ) / 1_000_000

    return {
        "model": model_id,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "cached_input_tokens": cached_input_tokens,
        "cost_usd": round(cost_usd, 8),
    }


def summarize_router_cost(result: dict[str, Any]) -> dict[str, Any]:
    cost = calculate_call_cost(result.get("usage_metadata", {}), result["served_by"])
    return {
        "served_by": result["served_by"],
        "trace_steps": len(result["trace"]),
        **cost,
    }


summarize_router_cost(router_result)

# 6 | Budget Gate

Ein Budget Gate entscheidet nicht, ob die Antwort fachlich richtig ist. Es entscheidet nur, ob der aktuelle Aufruf innerhalb der Kostenregel bleibt. In einer echten Pipeline wird diese Entscheidung mit Qualitäts- und Sicherheitsprüfungen kombiniert.


Das Ergebnis `budget_status` ist ein Evaluationssignal: Eine fachlich richtige Antwort kann im produktionsnahen Test trotzdem durchfallen, wenn Kosten, Tool-Anzahl oder Latenz aus dem Rahmen laufen. M24 nutzt solche Signale als zusätzliche Gates neben Antwortqualität und Quellenbindung.


In [ ]:
BUDGET_USD = 0.002


def route_with_budget(
    prompt: str,
    budget_usd: float,
    models: list[str] | None = None,
    forced_dead_models: set[str] | None = None,
) -> dict[str, Any]:
    breaker = CircuitBreaker(cooldown_seconds=60)
    result = route_llm(
        prompt=prompt,
        models=models or MODEL_CHAIN,
        breaker=breaker,
        forced_dead_models=forced_dead_models,
    )
    cost_summary = summarize_router_cost(result)
    cost_usd = cost_summary.get("cost_usd")

    if cost_usd is None:
        budget_status = "review_required"
    elif cost_usd <= budget_usd:
        budget_status = "ok"
    else:
        budget_status = "review_required"

    return {
        "content": result["content"],
        "served_by": result["served_by"],
        "budget_usd": budget_usd,
        "budget_status": budget_status,
        "cost": cost_summary,
        "trace": result["trace"],
    }


budget_result = route_with_budget(
    briefing_prompt,
    budget_usd=BUDGET_USD,
    forced_dead_models={MODEL_CHAIN[0]},
)

{
    "served_by": budget_result["served_by"],
    "budget_status": budget_result["budget_status"],
    "cost": budget_result["cost"],
}

# 7 | Optional: LangSmith-Kosten für ein Projekt auslesen

Wenn LangSmith für das Projekt aktiv ist, kann die Kostenkontrolle zusätzlich projektweit geprüft werden. Der folgende Code ist bewusst als optionale Hilfsfunktion formuliert, damit das Notebook auch ohne LangSmith-Auswertung lauffähig bleibt.


In [ ]:
from datetime import datetime, timedelta, timezone


def get_project_costs(project_name: str, days: int = 1) -> list[dict[str, Any]]:
    from langsmith import Client

    client = Client()
    start_time = datetime.now(timezone.utc) - timedelta(days=days)
    runs = client.list_runs(project_name=project_name, start_time=start_time)

    rows = []
    for run in runs:
        extra = getattr(run, "extra", {}) or {}
        usage = extra.get("usage_metadata") or extra.get("usage") or {}
        rows.append({
            "name": run.name,
            "run_type": run.run_type,
            "start_time": run.start_time,
            "usage": usage,
        })
    return rows


# Beispiel bei aktivem LangSmith-Projekt:
project_cost_rows = get_project_costs("M25-Model-Routing-Cost-Control", days=1)
project_cost_rows[:3]

# 8 | Anschluss an M26 und M37

| Zielmodul | Integration |
|---|---|
| M24 Agent Evaluation & Testing | `budget_status`, Kosten, Tool-Anzahl und Fehlerkategorien als Eval-Gates auswerten |
| M26 Integration Pipeline | Router vor den Briefing-Agenten setzen; Kostenstatus als eigenes Gate behandeln |
| M36 Production Deployment | Circuit-Breaker-Zustand und Fehlerkategorien in Monitoring/Alerting übernehmen |
| M37 API Deployment | `served_by`, Token und Budgetstatus in der API-Antwort oder im Trace protokollieren |
| M38 Capstone | Budgetregel als Akzeptanzkriterium für produktionsnahe Briefing-Workflows verwenden |


# A | Aufgaben
---

<p><font color='darkblue' size="4">
✏️  <b>Notiz:</b>
</font></p>

Die Aufgaben bauen eine kleine Betriebssicherung für den Meeting- & Research-Briefing-Agent. Jeder Selfcheck steht direkt unter der zugehörigen Umsetzung.


**Grundlagen**

Den Router-Trace prüfen und sichtbar machen, welches Modell den simulierten Ausfall übernommen hat.

**Erledigt wenn:** `mein_fallback_used` ist `True` und der Trace enthält mindestens einen `dead`-Eintrag.


In [ ]:
# Grundlagen: Router-Trace prüfen

# 1. mein_trace = router_result["trace"]
# 2. mein_fallback_used: router_result["served_by"] != MODEL_CHAIN[0]
# 3. mein_dead_events: Trace-Eintraege mit event == ErrorCategory.DEAD.value filtern

**Aufbau**

Token- und Kostendaten aus dem Router-Ergebnis zusammenfassen und prüfen, ob die Struktur für Monitoring geeignet ist.

**Erledigt wenn:** `mein_cost_summary` enthält Modell, Tokenfelder und `cost_usd`.


In [ ]:
# Aufbau: Kosten-Zusammenfassung prüfen

# 1. mein_cost_summary = summarize_router_cost(router_result)
# 2. Pflichtfelder prüfen: served_by, input_tokens, output_tokens, total_tokens, cost_usd

**Vertiefung**

Ein strenges Budget Gate simulieren, ohne einen weiteren Modellaufruf auszuführen.

**Erledigt wenn:** `strict_budget_status` steht auf `review_required`.


In [ ]:
# Vertiefung: strenges Budget Gate simulieren (ohne weiteren Modellaufruf)

# 1. strict_budget_usd = 0.0
# 2. cost_usd aus budget_result["cost"] lesen
# 3. strict_budget_status: "review_required" wenn cost_usd None oder > strict_budget_usd, sonst "ok"

<p><font color='darkblue' size="4">
✨ <b>Empfehlung:</b>
</font></p>

Für die Integration in M25 reicht ein kleines Betriebsgate: `served_by`, Trace, Tokenverbrauch, geschätzte Kosten und Budgetstatus werden protokolliert. Erst danach sollte die finale Briefing-Antwort freigegeben werden.



<p><font color='black' size="5">
Optionaler Trace-Check
</font></p>

Wenn LangSmith aktiv ist, kannst du die letzten Runs des Moduls anzeigen.


In [ ]:
#@markdown <p><font size="4" color='green'>LangSmith Trace-Analyse</font></p>

import time as _t
_t.sleep(2)
show_trace("M25-Model-Routing-Cost-Control", limit=5, show_steps=True)

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Modellauswahl](https://editor.p5js.org/ralf.bendig.rb/full/8BbTi8Ico)
- [Token-Verbrauch & Caching](https://editor.p5js.org/ralf.bendig.rb/full/uAPjZBhtW)
- [Modellsteuerung](https://editor.p5js.org/ralf.bendig.rb/full/um423ggnD)
- [Modellsteuerung Entscheidungshilfe](https://editor.p5js.org/ralf.bendig.rb/full/xb3zPgRSr)
- [LLM-Parameter](https://editor.p5js.org/ralf.bendig.rb/full/LBc3t3yP4)
- [Mixture of Experts](https://editor.p5js.org/ralf.bendig.rb/full/GT2XfGGTo)



# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Modellauswahl](https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/modellauswahl.html)
- [Provider-Modell-Mapping](https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/provider-modell-mapping.html)
- [LangSmith Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langsmith-best-practices.html)
- [Evaluation & Observability](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/evaluation-observability.html)
- [Minimum Viable Agent Stack](https://ralf-42.github.io/Agenten/08-deployment-betrieb/minimum-viable-agent-stack.html)
- [Digitale Souveränität](https://ralf-42.github.io/Agenten/09-regulatorik-verantwortung/digitale-souveraenitaet.html)
- [Migration-Analyse Provider](https://ralf-42.github.io/Agenten/08-deployment-betrieb/migration-openai-mistral.html)
- [Fine-Tuning](https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/fine-tuning.html)

